# 06 Evaluación y Métricas

RMSE para predicción y métricas de ranking: Precision@K, Recall@K, MAP, NDCG.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
base = Path(r'd:\GitHub\Ruta_aprendizaje_2024\03-aplicación_de_modelos\proyecto_3\data')
ratings = pd.read_csv(base / 'ratings.csv')

## División y ground truth

In [ ]:
np.random.seed(42)
mask = np.random.rand(len(ratings)) < 0.8
train = ratings[mask]
test = ratings[~mask]
gt = test.groupby('userId')['movieId'].apply(set)

## Recomendaciones dummy para ejemplo

In [ ]:
pop = train.groupby('movieId')['rating'].count().sort_values(ascending=False)
popular_items = list(pop.index)

## Métricas de ranking

In [ ]:
def precision_at_k(recs, truth, k=10):
    hits = len(set(recs[:k]) & truth)
    return hits / k
def recall_at_k(recs, truth, k=10):
    hits = len(set(recs[:k]) & truth)
    return hits / max(1, len(truth))
def dcg(scores):
    return np.sum([(s/np.log2(i+2)) for i,s in enumerate(scores)])
def ndcg_at_k(recs, truth, k=10):
    rel = [1 if r in truth else 0 for r in recs[:k]]
    ideal = sorted(rel, reverse=True)
    return dcg(rel) / max(1e-9, dcg(ideal))
def map_at_k(recs_list, truth_list, k=10):
    ap = []
    for recs, truth in zip(recs_list, truth_list):
        hits = 0
        s = 0
        for i,r in enumerate(recs[:k], start=1):
            if r in truth:
                hits += 1
                s += hits / i
        ap.append(s / max(1, len(set(recs[:k]) & truth)))
    return np.mean(ap)

## Evaluación con recomendaciones de popularidad

In [ ]:
users = list(gt.index)
recs_list = [popular_items for _ in users]
truth_list = [gt[u] for u in users]
p = np.mean([precision_at_k(recs_list[i], truth_list[i], k=10) for i in range(len(users))])
r = np.mean([recall_at_k(recs_list[i], truth_list[i], k=10) for i in range(len(users))])
n = np.mean([ndcg_at_k(recs_list[i], truth_list[i], k=10) for i in range(len(users))])
m = map_at_k(recs_list, truth_list, k=10)
p, r, n, m